# Graph-RAG Travel Chatbot — Learning Notebook (Google Colab)

**Goal:** Learn RAG and Graph-RAG end-to-end by building a tiny, working travel chatbot.

**What this chatbot does:**
1. Generates an itinerary for a place the user asks about
2. Answers factual questions about places
3. Suggests the best travel option with a budget comparison

**Stack:**
- Embeddings: `sentence-transformers`
- Vector DB: `ChromaDB`
- Graph: `networkx` - defines *relationships* between nodes
- LLM

## Concept Primer

**Plain RAG** = Retrieve relevant *text chunks* from a vector DB by similarity search → stuff them into the LLM prompt → LLM answers using that context instead of guessing.

##VS

**Graph RAG** = Same idea, but on top of similarity search we also *walk relationships between entities* (city → attraction, city → transport → city, attraction → category). This lets us pull in context that is **related but not textually similar** — e.g. a user asks about 'Paris' and we also retrieve 'Eiffel Tower' and 'flights to Paris' because the graph *knows* they're connected, even if their embeddings aren't close to the query.

End-to-end pipeline:

```
User query
   │
   ▼
[1] Embed query  ──────────────► sentence-transformers
   │
   ▼
[2] Vector similarity search ──► ChromaDB  (finds 'seed' nodes)
   │
   ▼
[3] Graph expansion (1-hop) ───► networkx  (pulls in connected nodes: attractions, transport, costs)
   │
   ▼
[4] Build context string  ─────► combine seed + expanded nodes' text
   │
   ▼
[5] Pick a prompt template ────► itinerary / place-info / budget-comparison
   │
   ▼
[6] Call LLM  ─────────────► generation grounded in retrieved context
   │
   ▼
Response to user
```


## Step 0 — Install dependencies (Colab)


In [25]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers networkx transformers accelerate requests openai


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [16]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Step 1 — Dummy Data (this is our 'knowledge base')

We model 4 kinds of entities (**nodes**):
- `city` — a destination
- `attraction` — a place to visit inside a city
- `transport` — a way to travel between two cities, with cost & duration
- `category` — a tag like 'landmark', 'nature', 'food' (used for graph relationships)

Each node has a short `description` string — that string is what gets embedded into the vector DB.


In [17]:
cities = [
    {"id": "paris", "name": "Paris", "country": "France",
     "description": "Paris is the capital of France, famous for art, fashion, romance, and the Eiffel Tower. Best visited Apr-Jun or Sep-Oct."},
    {"id": "tokyo", "name": "Tokyo", "country": "Japan",
     "description": "Tokyo is Japan's capital, blending ultra-modern skyscrapers with historic temples. Best visited Mar-May for cherry blossoms."},
    {"id": "goa", "name": "Goa", "country": "India",
     "description": "Goa is a coastal state in India known for beaches, nightlife, and Portuguese-era architecture. Best visited Nov-Feb."},
    {"id": "delhi", "name": "Delhi", "country": "India",
     "description": "Delhi is India's capital, home to Mughal-era monuments and a major international travel hub."},
]

attractions = [
    {"id": "eiffel_tower", "city_id": "paris", "name": "Eiffel Tower", "category": "landmark",
     "description": "Eiffel Tower: iconic iron tower in Paris, best visited at sunset, entry ~EUR 28, ~2 hours to explore."},
    {"id": "louvre", "city_id": "paris", "name": "Louvre Museum", "category": "culture",
     "description": "Louvre Museum: world's largest art museum in Paris, home to the Mona Lisa, entry ~EUR 22, allow half a day."},
    {"id": "senso_ji", "city_id": "tokyo", "name": "Senso-ji Temple", "category": "culture",
     "description": "Senso-ji: Tokyo's oldest Buddhist temple in Asakusa, free entry, best visited early morning to avoid crowds."},
    {"id": "shibuya", "city_id": "tokyo", "name": "Shibuya Crossing", "category": "landmark",
     "description": "Shibuya Crossing: world's busiest pedestrian crossing in Tokyo, free, best experienced at night."},
    {"id": "baga_beach", "city_id": "goa", "name": "Baga Beach", "category": "nature",
     "description": "Baga Beach: popular beach in North Goa known for water sports and beach shacks, free entry."},
    {"id": "fort_aguada", "city_id": "goa", "name": "Fort Aguada", "category": "landmark",
     "description": "Fort Aguada: 17th-century Portuguese fort in Goa with sea views, entry ~INR 25."},
    {"id": "red_fort", "city_id": "delhi", "name": "Red Fort", "category": "landmark",
     "description": "Red Fort: Mughal-era fortress in Delhi, UNESCO site, entry ~INR 35 for Indians / INR 550 for foreigners."},
]

transport_options = [
    {"id": "flight_del_par", "from_city": "delhi", "to_city": "paris", "mode": "flight",
     "cost_usd": 650, "duration_hours": 9,
     "description": "Flight from Delhi to Paris: ~9 hours, ~USD 650 round trip, most convenient option."},
    {"id": "flight_del_tokyo", "from_city": "delhi", "to_city": "tokyo", "mode": "flight",
     "cost_usd": 550, "duration_hours": 8,
     "description": "Flight from Delhi to Tokyo: ~8 hours, ~USD 550 round trip."},
    {"id": "flight_del_goa", "from_city": "delhi", "to_city": "goa", "mode": "flight",
     "cost_usd": 90, "duration_hours": 2.5,
     "description": "Flight from Delhi to Goa: ~2.5 hours, ~USD 90 round trip, fastest option."},
    {"id": "train_del_goa", "from_city": "delhi", "to_city": "goa", "mode": "train",
     "cost_usd": 35, "duration_hours": 26,
     "description": "Train from Delhi to Goa: ~26 hours, ~USD 35 one way, cheapest option, scenic but slow."},
    {"id": "bus_del_goa", "from_city": "delhi", "to_city": "goa", "mode": "bus",
     "cost_usd": 45, "duration_hours": 30,
     "description": "Bus from Delhi to Goa: ~30 hours, ~USD 45, budget option but long and tiring."},
]

categories = [
    {"id": "landmark", "name": "Landmark", "description": "Famous man-made structures and monuments."},
    {"id": "culture", "name": "Culture", "description": "Museums, temples, and cultural heritage sites."},
    {"id": "nature", "name": "Nature", "description": "Beaches, parks, and natural attractions."},
]

print(f"{len(cities)} cities, {len(attractions)} attractions, {len(transport_options)} transport options, {len(categories)} categories")


4 cities, 7 attractions, 5 transport options, 3 categories


## Step 2 — Build the Knowledge Graph (this is the 'Graph' in Graph-RAG)

We use `networkx` to define **explicit relationships** between nodes:
- `city --HAS_ATTRACTION--> attraction`
- `attraction --HAS_CATEGORY--> category`
- `city --TRANSPORT_TO--> transport_option --TRANSPORT_TO--> city`

These edges are what let us retrieve *related* information even when it isn't a close text match to the query.


In [18]:
import networkx as nx

G = nx.DiGraph()

def add_node(node_id, node_type, **attrs):
    G.add_node(node_id, type=node_type, **attrs)

for c in cities:
    add_node(c["id"], "city", name=c["name"], description=c["description"])

for cat in categories:
    add_node(cat["id"], "category", name=cat["name"], description=cat["description"])

for a in attractions:
    add_node(a["id"], "attraction", name=a["name"], description=a["description"])
    G.add_edge(a["city_id"], a["id"], relation="HAS_ATTRACTION")
    G.add_edge(a["id"], a["category"], relation="HAS_CATEGORY")

for t in transport_options:
    add_node(t["id"], "transport", name=f"{t['mode'].title()} {t['from_city']}->{t['to_city']}",
             description=t["description"], cost_usd=t["cost_usd"], duration_hours=t["duration_hours"], mode=t["mode"])
    G.add_edge(t["from_city"], t["id"], relation="TRANSPORT_TO")
    G.add_edge(t["id"], t["to_city"], relation="TRANSPORT_TO")

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print("Example neighbors of 'paris':", list(G.successors("paris")), list(G.predecessors("paris")))


Graph built: 19 nodes, 24 edges
Example neighbors of 'paris': ['eiffel_tower', 'louvre'] ['flight_del_par']


## Step 2B — Replace static text with REAL web content (Wikipedia) + Chunking

So far every node's `description` was a hand-written one-liner. Real RAG systems pull from actual sources — web pages, PDFs, wikis — which are far too long to embed as a single vector (see the chunking explanation above). This step:

1. **Fetches** a real Wikipedia article per city (plain-text extract via Wikipedia's API — no scraping/HTML parsing needed).
2. **Chunks** each article into overlapping word-windows (~80 words each, 15-word overlap) instead of embedding the whole article as one vector.
3. **Adds each chunk as its own graph node**, connected to its city with a new `HAS_CHUNK` edge — so the graph now has a *real*, multi-chunk knowledge source sitting alongside the small hand-written attraction/transport nodes.

Nothing downstream changes: Step 3's embedding loop iterates over *every* node in `G`, so these new chunk nodes get embedded and stored automatically the next time you run it. Step 4's graph walk already checks successors/predecessors generically, so `HAS_CHUNK` edges get followed the same way `HAS_ATTRACTION` edges do.


In [19]:
import requests

def fetch_wikipedia_extract(title, timeout=10):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "exintro": True,   # <-- only the lead section, not the entire article
        "titles": title,
        "format": "json",
    }
    headers = {"User-Agent": "GraphRAGTravelChatbot/1.0 (educational Colab notebook)"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=timeout)
        resp.raise_for_status()
        pages = resp.json()["query"]["pages"]
        return next(iter(pages.values())).get("extract", "")
    except Exception as e:
        print(f"  [!] Failed to fetch '{title}': {e}")
        return ""


In [20]:
def chunk_text(text, chunk_size_words=80, overlap_words=15):
    """Split long text into overlapping word-windows. This is the 'chunking' step:
    turns one huge string into many small, individually-embeddable pieces."""
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    step = max(chunk_size_words - overlap_words, 1)  # avoid infinite loop if overlap >= chunk_size
    while start < len(words):
        chunk_words = words[start:start + chunk_size_words]
        chunks.append(" ".join(chunk_words))
        start += step
    return chunks

# quick sanity check on a tiny example
demo = " ".join([f"word{i}" for i in range(20)])
print(chunk_text(demo, chunk_size_words=10, overlap_words=3))


['word0 word1 word2 word3 word4 word5 word6 word7 word8 word9', 'word7 word8 word9 word10 word11 word12 word13 word14 word15 word16', 'word14 word15 word16 word17 word18 word19']


In [21]:
# Map our internal city ids to real Wikipedia page titles
city_wiki_titles = {
    "paris": "Paris",
    "tokyo": "Tokyo",
    "goa": "Goa",
    "delhi": "Delhi",
}

total_chunks_added = 0

for city_id, title in city_wiki_titles.items():
    article_text = fetch_wikipedia_extract(title)
    if not article_text:
        print(f"{title}: no content fetched, skipping (city node still has its original hand-written description)")
        continue

    chunks = chunk_text(article_text, chunk_size_words=80, overlap_words=15)
    for idx, chunk_text_piece in enumerate(chunks):
        chunk_id = f"{city_id}_chunk_{idx}"
        add_node(
            chunk_id,
            "chunk",
            name=f"{title} (Wikipedia chunk {idx+1}/{len(chunks)})",
            description=chunk_text_piece,
        )
        G.add_edge(city_id, chunk_id, relation="HAS_CHUNK")

    total_chunks_added += len(chunks)
    print(f"{title}: fetched {len(article_text.split())} words -> {len(chunks)} chunks")

print(f"\nGraph now has {G.number_of_nodes()} nodes ({total_chunks_added} of them are Wikipedia chunks)")


Paris: fetched 387 words -> 6 chunks
Tokyo: fetched 616 words -> 10 chunks
Goa: fetched 315 words -> 5 chunks
Delhi: fetched 573 words -> 9 chunks

Graph now has 49 nodes (30 of them are Wikipedia chunks)


**Important:** because we just added nodes to `G` *after* Step 3's original embedding cell already ran once, you need to **re-run the Step 3 cell now** (the one that builds `all_nodes`, `documents`, `embeddings` and calls `collection.upsert(...)`) so the new chunk nodes get embedded and stored in Chroma too. `upsert` is safe to re-run — it just refreshes existing ids and adds new ones.


## Step 3 — Vector DB: embed every node and store it

**Role of the vector DB:** it lets us find nodes whose *description* is semantically similar to the user's query — even if the wording doesn't match exactly (e.g. 'cheap way to get to the beach state' should still match Goa transport options).

How it works:
1. Each node's `description` text is converted into a fixed-length numeric vector (embedding) using a local sentence-transformer model.
2. Chroma stores these vectors + the original text + metadata (node id, type).
3. At query time, the user's question is embedded the same way, and Chroma finds the nodes whose vectors are *closest* (cosine similarity) to the query vector.


In [22]:
import chromadb
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")  # small, free, runs locally/CPU

chroma_client = chromadb.Client()  # in-memory for this demo
collection = chroma_client.get_or_create_collection(name="travel_nodes")

all_nodes = list(G.nodes(data=True))
ids = [nid for nid, _ in all_nodes]
documents = [f"[{data['type']}] {data.get('name','')}: {data.get('description','')}" for _, data in all_nodes]
metadatas = [{"type": data["type"], "name": data.get("name", "")} for _, data in all_nodes]
embeddings = embed_model.encode(documents).tolist()

# Reset the collection first so it always exactly matches the CURRENT graph G
# (avoids stale node ids left over from an earlier run if you re-ran Step 2/2B before this cell)
chroma_client.delete_collection("travel_nodes")
collection = chroma_client.get_or_create_collection(name="travel_nodes")

collection.upsert(ids=ids, documents=documents, metadatas=metadatas, embeddings=embeddings)
print(f"Inserted {len(ids)} node embeddings into ChromaDB")


/home/bacancy/Desktop/Darsh-new/Learning/RAG-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4557.87it/s]


Inserted 49 node embeddings into ChromaDB


## Step 4 — Graph-RAG Retrieval Function

This is the core trick that separates Graph-RAG from plain RAG:
1. Vector search → get top-k **seed nodes** most similar to the query.
2. For each seed node, walk 1 hop in the graph (successors + predecessors) → **expanded nodes**.
3. Combine seed + expanded node text into one context block for the LLM.


In [23]:
def graph_rag_retrieve(query, top_k=3, hops=1, max_context_nodes=15, verbose=False):
    query_emb = embed_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=top_k)
    seed_ids = [nid for nid in results["ids"][0] if nid in G]

    expanded_ids = set(seed_ids)
    frontier = set(seed_ids)
    for _ in range(hops):
        next_frontier = set()
        for nid in frontier:
            if nid in G:
                next_frontier.update(G.successors(nid))
                next_frontier.update(G.predecessors(nid))
        expanded_ids.update(next_frontier)
        frontier = next_frontier

    # Cap total context size: keep all seeds, then prefer compact structured nodes
    # (attraction/transport/category) over generic text chunks when trimming.
    if len(expanded_ids) > max_context_nodes:
        non_seed = expanded_ids - set(seed_ids)
        prioritized = sorted(non_seed, key=lambda nid: 0 if G.nodes[nid]["type"] != "chunk" else 1)
        keep = max(max_context_nodes - len(seed_ids), 0)
        expanded_ids = set(seed_ids) | set(prioritized[:keep])

    context_lines = []
    for nid in expanded_ids:
        if nid not in G:
            continue
        data = G.nodes[nid]
        tag = "[SEED]" if nid in seed_ids else "[GRAPH-LINKED]"
        context_lines.append(f"{tag} ({data['type']}) {data.get('name','')}: {data.get('description','')}")

    if verbose:
        print("Seed nodes (vector search):", seed_ids)
        print("Expanded nodes (graph walk):", expanded_ids - set(seed_ids))

    return "\n".join(context_lines)

## Step 5 — Connect the LLM


In [26]:
from openai import OpenAI

groq_client = OpenAI(api_key="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", base_url="https://api.groq.com/openai/v1")

def call_llm(system_prompt, user_prompt, temperature=0.4, max_new_tokens=1200):
    resp = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=max_new_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content

In [27]:
models = groq_client.models.list()
for m in models.data:
    print(m.id)

qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
whisper-large-v3-turbo
canopylabs/orpheus-arabic-saudi
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
openai/gpt-oss-120b


## Step 6 — Prompt Templates

The prompts do three jobs:
1. **System prompt** — sets the chatbot's role and tells it to only use the retrieved context (grounding, reduces hallucination).
2. **Task-specific templates** — one each for itinerary, place-info, and budget comparison, so the LLM's output is structured and predictable.
3. **Context injection** — the Graph-RAG context string from Step 4 is inserted into the user prompt.


In [30]:
SYSTEM_PROMPT = (
    "You are a helpful travel assistant. Answer ONLY using the CONTEXT provided below. "
    "If the context does not contain enough information, say so honestly instead of making facts up. "
    "Be concise, practical, and use bullet points where helpful."
)

ITINERARY_TEMPLATE = """CONTEXT:
{context}

TASK: Create a day-by-day itinerary for the user's request below, using only attractions present in the context.
USER REQUEST: {query}"""

PLACE_INFO_TEMPLATE = """CONTEXT:
{context}

TASK: Answer the user's question about the place(s) using only the context above.
USER REQUEST: {query}"""

BUDGET_TEMPLATE = """CONTEXT:
{context}

TASK: Compare the available transport options in the context in a small markdown table (mode, cost, duration), then recommend the best option for a budget traveler and the best for a time-constrained traveler.
USER REQUEST: {query}"""


## Step 7 — Intent Router + Full Chatbot Function

A tiny keyword-based router decides which prompt template to use (itinerary / place-info / budget). In production you'd likely ask the LLM itself to classify intent, but a rule-based router keeps this demo simple and free of extra API calls.


In [31]:
def route_intent(query):
    q = query.lower()
    if any(w in q for w in ["itinerary", "plan", "day", "days", "trip plan", "schedule"]):
        return "itinerary"
    if any(w in q for w in ["cost", "budget", "cheap", "price", "flight", "train", "bus", "how to reach", "how to get", "travel option"]):
        return "budget"
    return "place_info"

TEMPLATES = {
    "itinerary": ITINERARY_TEMPLATE,
    "budget": BUDGET_TEMPLATE,
    "place_info": PLACE_INFO_TEMPLATE,
}

def travel_chatbot(query, top_k=4, hops=1, verbose=False):
    intent = route_intent(query)
    context = graph_rag_retrieve(query, top_k=top_k, hops=hops, verbose=verbose)
    prompt = TEMPLATES[intent].format(context=context, query=query)
    answer = call_llm(SYSTEM_PROMPT, prompt)
    if verbose:
        print(f"[intent detected: {intent}]\n")
    return answer


## Step 8 — Request-Response Chat Loop

This is the actual 'chatbot' — a simple loop, no UI, exactly what was asked for. Run the cell, type a query, type `exit` to stop.


In [32]:
def run_chat():
    print("Travel Chatbot (Graph-RAG demo). Type 'exit' to quit.\n")
    while True:
        user_query = input("You: ")
        if user_query.strip().lower() in {"exit", "quit"}:
            print("Bot: Safe travels! 👋")
            break
        answer = travel_chatbot(user_query, verbose=True)
        print(f"\nBot: {answer}\n")

# Uncomment to run interactively in Colab:
run_chat()


Travel Chatbot (Graph-RAG demo). Type 'exit' to quit.

Seed nodes (vector search): ['delhi', 'flight_del_tokyo', 'flight_del_par', 'flight_del_goa']
Expanded nodes (graph walk): {'delhi_chunk_6', 'delhi_chunk_1', 'red_fort', 'train_del_goa', 'bus_del_goa', 'delhi_chunk_0', 'delhi_chunk_2', 'goa', 'tokyo', 'paris', 'delhi_chunk_7'}
[intent detected: itinerary]


Bot: **7‑Day Delhi Itinerary (using only attractions mentioned in the context)**  

| Day | Activity | Notes |
|-----|----------|-------|
| **Day 1 – Arrival & Orientation** | • Check into hotel near Old Delhi or New Delhi.<br>• Light stroll around the Red Fort area (just outside the gate) to get a feel for the city. | No major attraction visit today – allows you to acclimate and rest. |
| **Day 2 – Red Fort** | • Full‑day visit to the Red Fort (UNESCO World Heritage Site).<br>• Entry fee: ~INR 35 for Indians, ~INR 550 for foreigners.<br>• Explore the palace halls, gardens, and the nearby Jama Masjid (if you wish to wander, thou

## Step 9 — Sample Test Calls

Run these to see the three core behaviors without using the interactive loop.


In [ ]:
print("=== ITINERARY ===")
print(travel_chatbot("Plan a 2 day itinerary for Paris", verbose=True))


In [ ]:
print("=== PLACE INFO ===")
print(travel_chatbot("Tell me about Senso-ji Temple in Tokyo", verbose=True))


In [ ]:
print("=== BUDGET COMPARISON ===")
print(travel_chatbot("What's the cheapest way to travel from Delhi to Goa?", verbose=True))


## Recap — What you just learned

| Concept | Where it happened |
|---|---|
| **Data configuration** | Step 1 — small hand-written records for attractions/transport/categories; Step 2B — real Wikipedia articles per city, split into overlapping chunks (`chunk_text()`) and added as graph nodes, since a whole article is too big to embed as one meaningful vector |
| **Vector DB role** | Step 3 — turns text into embeddings, enables *semantic* similarity search instead of exact keyword match |
| **What makes it 'Graph' RAG** | Step 2 + Step 4 — explicit `networkx` edges (HAS_ATTRACTION, TRANSPORT_TO, HAS_CATEGORY) let retrieval expand beyond pure similarity, pulling in *related* nodes |
| **LLM** | Step 5 — a free, open-source Hugging Face instruct model (`Qwen/Qwen2.5-1.5B-Instruct`), run locally via `transformers.pipeline`; swappable for a hosted free API or a paid provider like Grok with no change elsewhere |
| **Role of prompts** | Step 6 — a system prompt grounds the model in retrieved context only (reduces hallucination); task templates shape output format per use case |
| **Wiring it together** | Step 7-8 — `route_intent()` picks a template, `graph_rag_retrieve()` supplies context, `call_grok()` generates the answer, `run_chat()` is the request-response loop |

### Ideas to extend this once it works
- Swap the keyword-based `route_intent` for an LLM-based classifier.
- Add more hop levels (`hops=2`) and see how context grows — and starts including irrelevant nodes (a real graph-RAG tradeoff).
- Persist Chroma to disk (`chromadb.PersistentClient`) instead of in-memory so data survives between runs.
- Replace dummy data with a real dataset (e.g. scraped city guides) — the pipeline doesn't change, only Step 1.
- Try a real graph database (Neo4j) instead of `networkx` for larger, persistent graphs — the retrieval logic in Step 4 maps directly to a Cypher query.
